# Chapter 38 — Retrieval: Grounding a Model in Your Documents

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Create the data

Run once. Every dataset in this book is generated by code you can read — nothing is downloaded, so nothing can rot behind a dead link. This is the printed block from Chapter 38 (`code/ch38/gen_docs.py` in the repository).

In [2]:
policy = """
# Refunds
The refund window is 30 days from the delivery date. Items must be unused
and in original packaging. Refunds are issued to the original payment
method within 5 business days of our receiving the returned item.
Gift cards cannot be refunded after purchase under any circumstances.

# Shipping
Standard shipping takes 5 to 7 business days within the continental US.
Express shipping arrives in 2 business days if ordered before 2 PM.
We do not ship to PO boxes. International delivery takes 10 to 21 days
and any customs duties are payable by the recipient.

# Warranty
The warranty covers manufacturing defects for one year from purchase.
Damage from misuse, liquid, or unauthorized repair is excluded. A defective
unit is replaced rather than repaired when stock is available.

# Accounts
You may reset your password from the sign-in screen. Accounts inactive for
24 months are archived. To close an account, contact support; closure is
permanent and any store credit is forfeited.

# Payments
We accept major credit cards and bank transfer. Invoices for business
accounts are due 30 days from issue. A late fee of 1.5 percent per month
applies to overdue balances.
"""
open("policy.md", "w").write(policy.strip())
print(f"wrote policy.md: {len(policy.split())} words")

wrote policy.md: 195 words


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch38/_lib.py`.

In [3]:
import re, numpy as np, os as _os
# policy.md and tickets.txt are written by this chapter's own blocks (Step 1, gen_docs.py, and
# Step 5, gen_tickets.py), which the README's command runs in their printed places. If either
# block was skipped, write its file here so the later blocks still run: Step 1's line prints
# where the book prints it; Step 5's is written quietly so nothing appears out of order.
if not _os.path.exists("policy.md") and _os.path.exists("gen_docs.py"):
    exec(open("gen_docs.py").read())
if not _os.path.exists("tickets.txt") and _os.path.exists("gen_tickets.py"):
    import io as _io, contextlib as _cl
    with _cl.redirect_stdout(_io.StringIO()): exec(open("gen_tickets.py").read())
text = open("policy.md").read()
sections = re.split(r"\n(?=# )", text)
chunks = []
for sec in sections:
    title = sec.split("\n")[0].lstrip("# ").strip()
    body = " ".join(sec.split("\n")[1:]).strip()
    for sent in re.split(r"(?<=\.)\s+", body):
        if sent:
            chunks.append({"section": title, "text": sent})
texts = [c["text"] for c in chunks]

## The chapter code

### Block 1  (`c1.py`)

In [4]:
import re

text = open("policy.md").read()

# Chunk on headings, then on sentences if a section runs long.
sections = re.split(r"\n(?=# )", text)
chunks = []
for sec in sections:
    title = sec.split("\n")[0].lstrip("# ").strip()
    body = " ".join(sec.split("\n")[1:]).strip()
    for sent in re.split(r"(?<=\.)\s+", body):
        if sent:
            chunks.append({"section": title, "text": sent})

print(f"{len(sections)} sections -> {len(chunks)} chunks")
for c in chunks[:4]:
    print(f"  [{c['section']:<9}] {c['text'][:62]}...")

5 sections -> 17 chunks
  [Refunds  ] The refund window is 30 days from the delivery date....
  [Refunds  ] Items must be unused and in original packaging....
  [Refunds  ] Refunds are issued to the original payment method within 5 bus...
  [Refunds  ] Gift cards cannot be refunded after purchase under any circums...


### Block 2  (`c2.py`)

In [5]:
# Ten questions, with the index of the chunk that answers each.
queries = [
    ("How long do I have to send something back?",        0),
    ("When will the money reach my card?",                2),
    ("Can I get my money back on a gift voucher?",        3),
    ("How fast is the quickest delivery option?",         5),
    ("Do you deliver overseas and who pays the duty?",    7),
    ("What if my device stops working from a fault?",     8),
    ("Is water damage covered?",                          9),
    ("I forgot my login credentials.",                   11),
    ("What happens to a dormant profile?",               12),
    ("What is the penalty for paying an invoice late?",  16),
]

### Block 3  (`c3.py`)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

vec = TfidfVectorizer(stop_words="english").fit(texts)
index = np.asarray(normalize(vec.transform(texts)).todense())

def search(q, k=3):
    qv = np.asarray(normalize(vec.transform([q])).todense()).ravel()
    sims = index @ qv
    if sims.max() == 0:        # no shared vocabulary: nothing found
        return [], sims
    return list(np.argsort(-sims)[:k]), sims

hits = empty = 0
for q, gold in queries:
    top, sims = search(q)
    if not top:
        empty += 1
        print(f"  NONE  {q[:46]:<46}")
        continue
    ok = gold in top
    hits += ok
    print(f"  {'HIT ' if ok else 'MISS'}  {q[:46]:<46} "
          f"best={sims[top[0]]:.2f}")

print(f"\ntf-idf recall@3: {hits}/{len(queries)} = {hits/len(queries):.0%}"
      f"   ({empty} queries matched no vocabulary at all)")

  NONE  How long do I have to send something back?    
  NONE  When will the money reach my card?            
  HIT   Can I get my money back on a gift voucher?     best=0.47
  MISS  How fast is the quickest delivery option?      best=0.39
  NONE  Do you deliver overseas and who pays the duty?
  NONE  What if my device stops working from a fault? 
  HIT   Is water damage covered?                       best=0.41
  NONE  I forgot my login credentials.                
  NONE  What happens to a dormant profile?            
  HIT   What is the penalty for paying an invoice late best=0.38

tf-idf recall@3: 3/10 = 30%   (6 queries matched no vocabulary at all)


### The chapter's Step 5 block  (`gen_tickets.py`)

Writes `tickets.txt`, which the blocks below read. Printed here, where the book prints it.

In [7]:
import numpy as np
rng = np.random.default_rng(5)

# Past support conversations. Each topic has its own vocabulary, and the
# same idea gets phrased many ways inside it. That pairing -- different
# words, same company -- is the only signal an encoder needs.
topics = {
  "refund_window": (["return", "send back", "give back", "ship back"],
     ["thirty days", "delivery date", "unused",
      "original packaging", "window"]),
  "refund_timing": (["refund", "money back", "reimbursement", "repayment"],
     ["payment method", "business days", "issued", "card", "processed"]),
  "voucher":       (["gift card", "gift voucher", "store credit"],
     ["purchase", "cannot", "non refundable", "balance"]),
  "fast_ship":     (["express", "quickest", "fastest", "rush delivery"],
     ["two business days", "before 2 pm", "arrives", "option", "upgrade"]),
  "abroad":        (["international", "overseas", "outside the country"],
     ["customs", "duties", "recipient", "twenty one days", "border"]),
  "fault":         (["defect", "fault", "manufacturing flaw",
                     "stopped working"],
     ["warranty", "one year", "replaced", "unit", "covered"]),
  "liquid":        (["water damage", "liquid damage", "spill"],
     ["misuse", "excluded", "not covered", "unauthorized repair"]),
  "login":         (["password", "login credentials", "sign in details"],
     ["reset", "screen", "email", "account access", "forgot"]),
  "dormant":       (["inactive", "dormant", "unused for months", "idle"],
     ["archived", "twenty four months", "account", "reactivate"]),
  "overdue":       (["late fee", "penalty for paying late", "overdue charge"],
     ["invoice", "1.5 percent", "monthly", "outstanding balance"]),
}

keys = list(topics)
generic = ["order", "account", "please", "team", "ticket", "policy"]
rows = []
for _ in range(6000):
    key = str(rng.choice(keys))
    terms, ctx = topics[key]
    # synonyms are not perfectly interchangeable: each has its own slight tilt
    t_i = int(rng.integers(len(terms)))
    term = terms[t_i]
    # each phrasing leans on the topic's context words a little differently
    w = np.full(len(ctx), 1.0)
    w[t_i % len(ctx)] += 2.2
    w[(t_i + 1) % len(ctx)] += 1.1
    words = [term] + list(rng.choice(ctx, 3, replace=False, p=w / w.sum()))
    if rng.random() < 0.18:                      # conversations wander
        other = topics[str(rng.choice(keys))][1]
        words.append(str(rng.choice(other)))
    words += list(rng.choice(generic, 2, replace=False))
    rng.shuffle(words)
    rows.append("customer asked about " + " ".join(words))

open("tickets.txt", "w").write("\n".join(rows))
print(f"wrote tickets.txt: {len(rows):,} past conversations across "
      f"{len(topics)} topics")

wrote tickets.txt: 6,000 past conversations across 10 topics


### Block 4  (`c4.py`)

In [8]:
from collections import Counter
import numpy as np

# Count which words appear near which other words.
lines = [l.split() for l in open("tickets.txt").read().split("\n")]
vocab = sorted({w for l in lines for w in l})
ix = {w: i for i, w in enumerate(vocab)}
C = np.zeros((len(vocab), len(vocab)))
for l in lines:
    for i, w in enumerate(l):
        for u in l[max(0, i-4):i+5]:
            if u != w:
                C[ix[w], ix[u]] += 1

# Positive pointwise mutual information: how much more often two words
# appear together than chance would give.
tot = C.sum()
row, col = C.sum(1, keepdims=True), C.sum(0, keepdims=True)
pmi = np.log((C * tot) / (row * col + 1e-9) + 1e-9)
P = np.maximum(pmi, 0)

U, S, _ = np.linalg.svd(P, full_matrices=False)
E = U[:, :32] * S[:32]                       # 32-dimensional word vectors
E /= (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)

def cos(a, b):
    return E[ix[a]] @ E[ix[b]]

for a, b in [("dormant", "inactive"), ("overseas", "international"),
             ("express", "fastest"), ("dormant", "express")]:
    print(f"  cos({a:<12}, {b:<14}) = {cos(a, b):+.3f}")

  cos(dormant     , inactive      ) = +1.000
  cos(overseas    , international ) = +1.000
  cos(express     , fastest       ) = +0.999
  cos(dormant     , express       ) = +0.003


### Block 5  (`c5.py`)

In [9]:
import re
def embed(s):
    ws = [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w in ix]
    if not ws:
        return np.zeros(E.shape[1])
    v = E[[ix[w] for w in ws]].mean(0)
    n = np.linalg.norm(v)
    return v / n if n else v

dense = np.vstack([embed(t) for t in texts])

def search(q, k=3):
    sims = dense @ embed(q)
    return list(np.argsort(-sims)[:k]), sims

hits = 0
for q, gold in queries:
    top, sims = search(q)
    ok = gold in top
    hits += ok
    print(f"  {'HIT ' if ok else 'MISS'}  {q[:46]:<46} "
          f"best={sims[top[0]]:.2f}")
print(f"\ndense recall@3: {hits}/{len(queries)} = {hits/len(queries):.0%}"
      f"   (tf-idf managed 30%)")

  HIT   How long do I have to send something back?     best=0.83
  HIT   When will the money reach my card?             best=0.86
  HIT   Can I get my money back on a gift voucher?     best=0.73
  HIT   How fast is the quickest delivery option?      best=0.85
  HIT   Do you deliver overseas and who pays the duty? best=0.85
  HIT   What if my device stops working from a fault?  best=0.64
  HIT   Is water damage covered?                       best=0.95
  HIT   I forgot my login credentials.                 best=0.90
  HIT   What happens to a dormant profile?             best=0.84
  HIT   What is the penalty for paying an invoice late best=0.90

dense recall@3: 10/10 = 100%   (tf-idf managed 30%)


### Block 6  (`c6.py`)

In [10]:
# A retriever always returns its top k, even when the answer is absent.
out_of_scope = ["Do you offer a student discount?",
                "Who is the chief executive?",
                "Can I book an installation visit?"]

def head(text, n=24):        # a chunk's first words, cut at a word
    return text[:n].rsplit(" ", 1)[0] + " ..."

print(f"{'score':>6}  {'question':<44} top chunk retrieved")
for q, _ in queries[:2]:
    s = dense @ embed(q)
    print(f"{s.max():>6.2f}  {q:<44} {head(texts[s.argmax()])}")
print()
for q in out_of_scope:
    s = dense @ embed(q)
    print(f"{s.max():>6.2f}  {q:<44} {head(texts[s.argmax()])}")

THRESH = 0.55
bad = [q for q in out_of_scope if (dense @ embed(q)).max() >= THRESH]
print(f"\na floor at {THRESH} rejects {3 - len(bad)} of 3 out-of-scope "
      f"questions, and still lets through:")
for q in bad:
    print(f"  {q}")

 score  question                                     top chunk retrieved
  0.83  How long do I have to send something back?   Items must be unused ...
  0.86  When will the money reach my card?           Refunds are issued to ...

  0.00  Do you offer a student discount?             The refund window is 30 ...
  0.87  Who is the chief executive?                  International delivery ...
  0.00  Can I book an installation visit?            The refund window is 30 ...

a floor at 0.55 rejects 2 of 3 out-of-scope questions, and still lets through:
  Who is the chief executive?
